# Experiment 1 — fact + dimension, three strategies

`orders_big` (24 M rows, 16 files) joined with `customers` (10,000 rows, 1 file) on `customer_id`.
Each run computes the full join and writes nothing (`noop` sink). Spark UI: http://localhost:4040 → **SQL / DataFrame**, **Stages**.

In [1]:
import time
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("m4-exp1-join-strategies").getOrCreate()
fact = spark.read.parquet("/opt/data/m3/orders_big")
dim = spark.read.parquet("/opt/data/m1/customers")

def timed(label, df):
    t0 = time.perf_counter()
    df.write.format("noop").mode("overwrite").save()
    print(f"{label}: {time.perf_counter() - t0:.1f}s")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/29 09:05:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## A — no hints

**PREDICT:** join strategy; how many shuffle `Exchange` nodes and on which side(s); jobs and stages.

In [2]:
a = fact.join(dim, "customer_id")
a.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildRight (6)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan parquet  (3)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(3) Scan parquet 
Output [3]: [customer_id#6, country#7, segment#8]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m1/customers]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<customer_id:int,country:string,segment:string>

(4) Filter
Input [3]: [customer_id#6, country#7, segment#8

In [3]:
timed("A default", a)
timed("A default (warm)", a)

A default: 3.5s


[Stage 5:===================================>                     (10 + 6) / 16]

A default (warm): 1.1s


## B — `merge` hint

**PREDICT:** join node; `Exchange` and `Sort` nodes and on which sides; shuffle write per side; runtime relative to A.

In [4]:
b = fact.join(dim.hint("merge"), "customer_id")
b.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- SortMergeJoin Inner (9)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan parquet  (1)
      +- Sort (8)
         +- Exchange (7)
            +- Filter (6)
               +- Scan parquet  (5)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(3) Exchange
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: hashpartitioning(customer_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=218]

(4) Sort
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amou

In [5]:
timed("B merge", b)
timed("B merge (warm)", b)

B merge: 4.5s


[Stage 15:=============================================>          (18 + 4) / 22]

B merge (warm): 3.7s


## C — `shuffle_hash` hint

**PREDICT:** what differs from B in the plan; which side becomes the build side; runtime relative to A and B.

In [6]:
c = fact.join(dim.hint("shuffle_hash"), "customer_id")
c.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- ShuffledHashJoin Inner BuildRight (7)
      :- Exchange (3)
      :  +- Filter (2)
      :     +- Scan parquet  (1)
      +- Exchange (6)
         +- Filter (5)
            +- Scan parquet  (4)


(1) Scan parquet 
Output [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Batched: true
Location: InMemoryFileIndex [file:/opt/data/m3/orders_big]
PushedFilters: [IsNotNull(customer_id)]
ReadSchema: struct<order_id:bigint,customer_id:int,shop_id:int,amount:double,order_date:date,note:string>

(2) Filter
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Condition : isnotnull(customer_id#1)

(3) Exchange
Input [6]: [order_id#0L, customer_id#1, shop_id#2, amount#3, order_date#4, note#5]
Arguments: hashpartitioning(customer_id#1, 200), ENSURE_REQUIREMENTS, [plan_id=511]

(4) Scan parquet 
Output [3]: [customer_id#6, country#7, segment#8]
Batched: true
Location: InMemoryFileIn

In [7]:
timed("C shuffle_hash", c)
timed("C shuffle_hash (warm)", c)

C shuffle_hash: 3.0s


[Stage 25:===========================================>            (17 + 4) / 22]

C shuffle_hash (warm): 2.9s


## Observe

SQL / DataFrame tab for each query: join node, every `Exchange` / `BroadcastExchange` / `Sort`. Stages tab: shuffle write per map stage.

## Stop before opening the next notebook

In [ ]:
spark.stop()